# Part 1. Acquisition and Storage
## Notebook B — Storage layout, controlled benchmark and report evidence

**Input.** Verified Financials partitions and reference-table metadata generated by Notebook A in this study's Drive directory.

**Workflow.** 1 restores the runtime; 2 checks the acquisition contract; 3 constructs equivalent flat CSV/Parquet files; 4 measures their storage footprint; 5 benchmarks defined read workloads; 6 independently audits trial results; 7 presents tables and figures; 8 verifies that the final evidence traces back to the acquisition manifest. The input dataset is not modified.


### 1. Code, packages and storage-stage tests

A new Colab runtime requires its own package installation and Drive mount. Only benchmark, analysis and handoff tests are executed here; the acquisition is not repeated. Pinned versions and a fixed benchmark seed make the protocol explicit.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY = "https://github.com/Duongnb58/BFT65B_11230945_Final.git"
REPO_DIR = Path("/content/BFT65B_11230945_Final")
if REPO_DIR.is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPOSITORY, str(REPO_DIR)], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements.txt")],
    check=True,
)

TEST_XML = Path("/content/part1_storage_tests.xml")
subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-rs",
     "--junitxml", str(TEST_XML),
     str(REPO_DIR / "tests" / "test_storage_benchmark.py"),
     str(REPO_DIR / "tests" / "test_storage_analysis.py"),
     str(REPO_DIR / "tests" / "test_workflow.py")],
    cwd=REPO_DIR, check=True,
)

import xml.etree.ElementTree as ET
suites = ET.parse(TEST_XML).getroot().findall(".//testsuite")
assert suites and all(int(s.get("failures", 0)) == 0 and
                      int(s.get("errors", 0)) == 0 and
                      int(s.get("skipped", 0)) == 0 for s in suites),     "Storage tests must pass without skipped PyArrow integration checks"

sys.path.insert(0, str(REPO_DIR / "src"))
print("Storage-stage tests: passed, 0 skipped")


### 2. Verified acquisition input

The acquisition manifest and verification report establish the exact reporting period and observed row count. Their SHA-256 digests are carried into benchmark preparation and later checked against the final evidence. Results from other Drive directories are not considered.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from fdic_pipeline.workflow import StudyPaths, acquisition_contract

DRIVE_ROOT = Path("/content/drive/MyDrive/BFT65B_11230945_Final")
PATHS = StudyPaths(DRIVE_ROOT)
DATASET_DIR = PATHS.acquisition
ACQUISITION = acquisition_contract(DATASET_DIR)

print("Input directory:", DATASET_DIR)
print("Verified quarters:", ACQUISITION["quarters"])
print("Financials rows:", f"{ACQUISITION['financial_rows']:,}")
print("Manifest SHA-256:", ACQUISITION["manifest_sha256"])
print("Verification SHA-256:", ACQUISITION["verification_report_sha256"])


### 3. Controlled CSV and Parquet comparison inputs

The 170 source Parquet partitions are copied to local Colab storage and checked against the manifest. A flat uncompressed CSV and a flat Snappy Parquet file are then written from the same Arrow batches. Equal schema, row counts, values and null masks are required before any timing measurement.


In [ ]:
import json
import shutil

LOCAL_WORK = Path("/content/bft65b_11230945_storage")
free_gib = shutil.disk_usage("/content").free / (1024 ** 3)
print(f"Free local storage: {free_gib:.2f} GiB")
assert free_gib >= 4.0, "At least 4 GiB of local disk space is required"

PREPARE_CMD = [
    sys.executable, str(REPO_DIR / "run_storage_benchmark.py"), "prepare",
    "--source-dir", str(DATASET_DIR),
    "--work-dir", str(LOCAL_WORK),
]
subprocess.run(PREPARE_CMD, cwd=REPO_DIR, check=True)

with (LOCAL_WORK / "preparation.json").open(encoding="utf-8") as f:
    PREPARATION = json.load(f)
assert PREPARATION["round_trip_equality"] is True
assert PREPARATION["financial_rows"] == ACQUISITION["financial_rows"]
assert PREPARATION["num_quarters"] == ACQUISITION["quarters"]
assert PREPARATION["acquisition_manifest_sha256"] == ACQUISITION["manifest_sha256"]
assert PREPARATION["acquisition_report_sha256"] == ACQUISITION["verification_report_sha256"]
print("Format equality: passed")
print("Rows compared:", f"{PREPARATION['financial_rows']:,}")


### 4. Storage footprint

Three representations cover the same Financials scope: uncompressed flat CSV, flat Snappy Parquet, and the production quarter-partitioned Snappy Parquet layout. The reduction percentage below uses CSV and flat Parquet only; the partitioned layout is reported separately.


In [ ]:
MIB = 1024 ** 2
csv_mib = PREPARATION["csv_bytes"] / MIB
flat_mib = PREPARATION["flat_parquet_bytes"] / MIB
partitioned_mib = PREPARATION["production_partitioned_parquet_bytes"] / MIB
reduction = 100 * (1 - PREPARATION["flat_parquet_bytes"] / PREPARATION["csv_bytes"])
print(f"Uncompressed flat CSV       : {csv_mib:.3f} MiB")
print(f"Flat Snappy Parquet         : {flat_mib:.3f} MiB")
print(f"170-partition Snappy Parquet: {partitioned_mib:.3f} MiB")
print(f"Flat Parquet reduction      : {reduction:.2f}%")


### 5. Read benchmark: fresh-process trials

The experiment measures a full scan, six-column projection and reporting-date filter for both flat formats. Each workload/format receives one warmup and five measured runs. Each read occurs in a separate Python process; execution order is randomized with seed 11230945. Workload output rows and aggregates must match across formats. The benchmark and the following audit share one newly generated run ID.


In [ ]:
from datetime import datetime, timezone

RUN_ID = datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ")
BENCHMARK_DIR = PATHS.benchmark_root / RUN_ID
ANALYSIS_DIR = PATHS.analysis_root / RUN_ID
assert not BENCHMARK_DIR.exists(), f"Run ID already exists: {RUN_ID}"

BENCHMARK_CMD = [
    sys.executable, str(REPO_DIR / "run_storage_benchmark.py"), "benchmark",
    "--work-dir", str(LOCAL_WORK),
    "--results-dir", str(BENCHMARK_DIR),
    "--repetitions", "5", "--warmups", "1", "--seed", "11230945",
]
subprocess.run(BENCHMARK_CMD, cwd=REPO_DIR, check=True)

with (BENCHMARK_DIR / "benchmark_verification.json").open(encoding="utf-8") as f:
    BENCHMARK_VERIFICATION = json.load(f)
assert BENCHMARK_VERIFICATION["success"] is True
assert BENCHMARK_VERIFICATION["round_trip_equality"] is True
assert BENCHMARK_VERIFICATION["trial_count"] == 36
assert BENCHMARK_VERIFICATION["expected_rows"] == ACQUISITION["financial_rows"]
print("Benchmark run ID:", RUN_ID)
print("Validated measured and warmup trials:", BENCHMARK_VERIFICATION["trial_count"])


### 6. Independent analysis of measured results

An independent audit re-derives medians, speedups, sizes and trial coverage from the saved CSV tables. Warmups are excluded from medians. Inconsistent trial results or absent evidence stop report generation. Tables, figures and English narrative are saved under the same run ID in `storage_analysis`.


In [ ]:
ANALYSIS_CMD = [
    sys.executable, str(REPO_DIR / "run_storage_analysis.py"),
    "--benchmark-dir", str(BENCHMARK_DIR),
    "--output-dir", str(ANALYSIS_DIR),
]
subprocess.run(ANALYSIS_CMD, cwd=REPO_DIR, check=True)

with (ANALYSIS_DIR / "analysis_validation.json").open(encoding="utf-8") as f:
    ANALYSIS_VALIDATION = json.load(f)
assert ANALYSIS_VALIDATION["success"] is True
assert ANALYSIS_VALIDATION["benchmark_run"] == RUN_ID
assert ANALYSIS_VALIDATION["source_rows"] == ACQUISITION["financial_rows"]
assert ANALYSIS_VALIDATION["total_trials_verified"] == 36
print("Storage analysis: passed")
print("Flat Parquet storage reduction (%):",
      round(ANALYSIS_VALIDATION["statistics"]["flat_parquet_reduction_pct"], 2))


### 7. Tables, figures and interpretation

The displayed statistics are read from the audited evidence files, not typed into the notebook. Figure size is adjusted only for notebook readability; the original high-resolution figures remain in the evidence directory.


In [ ]:
from IPython.display import display, Image, Markdown, HTML
import csv

for title, filename in (
    ("Table 1. Storage footprint", "report_storage_table.csv"),
    ("Table 2. Read latency and memory", "report_performance_table.csv"),
):
    display(Markdown("**" + title + "**"))
    with (ANALYSIS_DIR / filename).open(newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    columns = list(rows[0])
    header = " | ".join(columns)
    divider = " | ".join("---" for _ in columns)
    body = [" | ".join(str(r[name]) for name in columns) for r in rows]
    display(Markdown("\n".join([header, divider, *body])))

for caption, image_name in (
    ("Figure 1. Storage footprint (MiB)", "storage_sizes.png"),
    ("Figure 2. Median read time (s)", "read_time_medians.png"),
    ("Figure 3. Median peak process RSS (MiB)", "peak_rss_medians.png"),
):
    display(Markdown("**" + caption + "**"))
    display(Image(filename=str(ANALYSIS_DIR / "figures" / image_name), width=680))
    display(HTML('<div style="height:22px"></div>'))

print("English evidence narrative:", ANALYSIS_DIR / "part1_storage_report_draft.md")


### 8. End-to-end evidence and report handoff

The final contract verifies that this analysis references the acquisition manifest and verification report used during preparation; its run ID matches the benchmark; every original evidence file still matches its recorded SHA-256; and the report artifacts are present. The resulting file locations support the Part 1 tables and figures in the final report. The local temporary comparison files may be removed after evidence has been retained on Drive.


In [ ]:
from fdic_pipeline.workflow import evidence_contract

EVIDENCE = evidence_contract(DATASET_DIR, BENCHMARK_DIR, ANALYSIS_DIR)
assert EVIDENCE["success"]
print("Part 1 evidence: PASSED")
print("Run ID:", EVIDENCE["run_id"])
print("Financials rows:", f"{EVIDENCE['acquisition']['financial_rows']:,}")
print("Trial executions:", EVIDENCE["measured_trials"])
print("Acquisition evidence:", DATASET_DIR / "verification_report.json")
print("Benchmark evidence:", BENCHMARK_DIR)
print("Report evidence:", ANALYSIS_DIR)
print("Git commit:", EVIDENCE["benchmark_environment"].get("git_commit_sha"))
